# GINO Evolution With Shared FNO Latent and MLP Super-Resolution Decoder

Trains a two-head learned simulator on the particle evolution dataset. The shared latent path is `GNO encoder -> FNO processor`; a GNO decoder predicts 10 particle update channels `[delta x, delta Gamma, delta sigma, delta u]`, and an MLP decoder reconstructs velocity `u(x)` at arbitrary query coordinates sampled from the latent grid.


In [ ]:
from __future__ import annotations

RUN_TAG = "gino_evolution_mlp_sr"
RESULTS_SUBDIR = "result/task1_gino_evolution_mlp_sr"

import inspect
import json
import math
import os
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    GNOBlock = None
    GNO_IMPORT_ERROR = error

try:
    from neuralop.models import FNO
except Exception as error:
    FNO = None
    FNO_IMPORT_ERROR = error

SEED = int(os.environ.get("EVOLUTION_SR_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _repo_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    if (cwd / "GINO_evolution").is_dir():
        return cwd, cwd
    if cwd.name == "GINO_evolution":
        return cwd.parent, cwd.parent
    if (cwd / "FINAL").is_dir():
        return cwd, cwd / "FINAL"
    if cwd.name == "FINAL":
        return cwd.parent, cwd
    return cwd, cwd


def _int_or_none(raw: str) -> Optional[int]:
    raw = str(raw).strip().lower()
    if raw in {"", "none", "null", "all", "0"}:
        return None
    return int(raw)


def _csv_env(name: str, default: Sequence[str]) -> list[str]:
    raw = os.environ.get(name, "").strip()
    return [x.strip() for x in raw.split(",") if x.strip()] if raw else list(default)

REPO_ROOT, FINAL_DIR = _repo_paths()
DATASET_CANDIDATES = [
    REPO_ROOT / "processed_data" / "particle_evolution_dataset.npz",
    REPO_ROOT / "process_data_evolution" / "particle_evolution_dataset.npz",
    FINAL_DIR / "processed_data" / "particle_evolution_dataset.npz",
    FINAL_DIR / "processed_data_evolution" / "particle_evolution_dataset.npz",
    FINAL_DIR / "process_data_evolution" / "particle_evolution_dataset.npz",
]
DATASET_PATH = Path(os.environ.get("EVOLUTION_DATASET", "")).expanduser()
if str(DATASET_PATH) in {"", "."}:
    DATASET_PATH = next((candidate for candidate in DATASET_CANDIDATES if candidate.exists()), DATASET_CANDIDATES[0])
if not DATASET_PATH.is_absolute():
    DATASET_PATH = (Path.cwd() / DATASET_PATH).resolve()
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing particle_evolution_dataset.npz: {DATASET_PATH}")

RESULTS_DIR = REPO_ROOT / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / f"{RUN_TAG}_best_model.pt"
HISTORY_PATH = RESULTS_DIR / f"{RUN_TAG}_history.json"

DEFAULT_INPUT_CHANNELS = [
    "Gamma_x", "Gamma_y", "Gamma_z", "sigma",
    "u_x", "u_y", "u_z",
    "geom_dist", "geom_nx", "geom_ny", "geom_nz",
    "angle_of_attack", "phase",
]
CFG = {
    "seed": SEED,

    "run_tag": RUN_TAG,

    "epochs": int(os.environ.get("EVOLUTION_SR_EPOCHS", "60")),

    "lr": float(os.environ.get("EVOLUTION_SR_LR", "3e-4")),

    "weight_decay": float(os.environ.get("EVOLUTION_SR_WEIGHT_DECAY", "3e-5")),

    "eval_every": int(os.environ.get("EVOLUTION_SR_EVAL_EVERY", "2")),

    "batch_size": 1,

    "num_workers": int(os.environ.get("EVOLUTION_SR_NUM_WORKERS", "0")),

    "maximum_input_particles": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_INPUT_PARTICLES", "4096")),

    "maximum_train_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_TRAIN_QUERY_POINTS", "2048")),

    "maximum_eval_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_EVAL_QUERY_POINTS", "12000")),

    "maximum_eval_batches": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_EVAL_BATCHES", "32")),

    "gradient_accumulation_steps": int(os.environ.get("EVOLUTION_SR_ACCUM_STEPS", "4")),

    "grad_clip_norm": float(os.environ.get("EVOLUTION_SR_GRAD_CLIP", "1.0")),

    "latent_res": int(os.environ.get("EVOLUTION_SR_LATENT_RES", "8")),

    "hidden_channels": int(os.environ.get("EVOLUTION_SR_HIDDEN", "96")),

    "fno_layers": int(os.environ.get("EVOLUTION_SR_FNO_LAYERS", "4")),

    "fno_modes": int(os.environ.get("EVOLUTION_SR_FNO_MODES", "6")),

    "gno_radius": float(os.environ.get("EVOLUTION_SR_GNO_RADIUS", "0.12")),

    "mlp_layers": int(os.environ.get("EVOLUTION_SR_MLP_LAYERS", "3")),

    "mlp_hidden": int(os.environ.get("EVOLUTION_SR_MLP_HIDDEN", "128")),

    "query_pos_encoding_frequencies": int(os.environ.get("EVOLUTION_SR_QUERY_PE_FREQS", "4")),

    "loss_weighting": os.environ.get("EVOLUTION_SR_LOSS_WEIGHTING", "uncertainty"),

    "field_loss_weight": float(os.environ.get("EVOLUTION_SR_FIELD_WEIGHT", "1.0")),

    "sr_grid_resolution": int(os.environ.get("EVOLUTION_SR_GRID_RES", "96")),
    
    "input_channels": _csv_env("EVOLUTION_SR_INPUT_CHANNELS", DEFAULT_INPUT_CHANNELS),
}

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_DIR)
print("Device :", DEVICE)
print(json.dumps(CFG, indent=2))


In [ ]:
dataset_file = np.load(DATASET_PATH, allow_pickle=True)
feature_names_all = [str(x) for x in dataset_file["feature_names"].tolist()]
target_names = [str(x) for x in dataset_file["target_names"].tolist()]
field_target_names = [str(x) for x in dataset_file["field_target_names"].tolist()] if "field_target_names" in dataset_file.files else ["u_x", "u_y", "u_z"]
expected_delta_targets = ["dx", "dy", "dz", "dGamma_x", "dGamma_y", "dGamma_z", "dsigma", "delta_u_x", "delta_u_y", "delta_u_z"]
if target_names != expected_delta_targets:
    raise RuntimeError(f"Expected 10-channel delta target {expected_delta_targets}; got {target_names}")
if field_target_names != ["u_x", "u_y", "u_z"]:
    raise RuntimeError(f"Expected velocity-field targets ['u_x','u_y','u_z']; got {field_target_names}")

missing_channels = [name for name in CFG["input_channels"] if name not in feature_names_all]
if missing_channels:
    raise KeyError(f"Missing input channels {missing_channels}; available={feature_names_all}")
for required_key in ["query_coords", "targets_velocity_field", "targets_velocity_field_norm", "field_query_mask"]:
    if required_key not in dataset_file.files:
        raise KeyError(f"Missing {required_key!r}. Regenerate the dataset with the updated processdata_evolution_u.py")

active_input_feature_indices = [feature_names_all.index(name) for name in CFG["input_channels"]]
coord_feature_indices = [feature_names_all.index(name) for name in ("x", "y", "z")]
feature_names = [feature_names_all[i] for i in active_input_feature_indices]

frame_contexts = list(dataset_file["pair_contexts"] if "pair_contexts" in dataset_file.files else dataset_file["frame_contexts"])
frame_ranges = list(dataset_file["pair_ranges"] if "pair_ranges" in dataset_file.files else dataset_file["frame_ranges"])
# Load each large compressed array exactly once. Avoid building per-pair copies from the
# 2+ GB .npz; repeated npz reads can make the Jupyter kernel disappear with no traceback.
inputs_t = np.asarray(dataset_file["inputs_t"], dtype=np.float32)
targets_delta_norm_all = np.asarray(dataset_file["targets_delta_norm"], dtype=np.float32)
query_coords_all = np.asarray(dataset_file["query_coords"], dtype=np.float32)
targets_field_norm_all = np.asarray(dataset_file["targets_velocity_field_norm"], dtype=np.float32)
field_query_mask_all = np.asarray(dataset_file["field_query_mask"], dtype=bool)

train_pair_ids = np.asarray(dataset_file["train_pair_ids"], dtype=np.int64)
val_pair_ids = np.asarray(dataset_file["val_pair_ids"], dtype=np.int64) if "val_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
test_pair_ids = np.asarray(dataset_file["test_pair_ids"], dtype=np.int64) if "test_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
if len(val_pair_ids) == 0 and len(train_pair_ids) > 4:
    val_pair_ids = train_pair_ids[2::5]
    val_set = set(int(i) for i in val_pair_ids)
    train_pair_ids = np.asarray([i for i in train_pair_ids if int(i) not in val_set], dtype=np.int64)

input_mean_all = np.asarray(dataset_file["in_mean"], dtype=np.float32).reshape(-1)
input_std_all = np.maximum(np.asarray(dataset_file["in_std"], dtype=np.float32).reshape(-1), 1e-8)
input_mean = input_mean_all[active_input_feature_indices]
input_std = input_std_all[active_input_feature_indices]
target_mean = np.asarray(dataset_file["out_mean"], dtype=np.float32).reshape(-1)
target_std = np.maximum(np.asarray(dataset_file["out_std"], dtype=np.float32).reshape(-1), 1e-8)
field_mean = np.asarray(dataset_file["field_mean"], dtype=np.float32).reshape(-1)
field_std = np.maximum(np.asarray(dataset_file["field_std"], dtype=np.float32).reshape(-1), 1e-8)
if not (np.isfinite(field_mean).all() and np.isfinite(field_std).all()):
    raise RuntimeError(
        "Field normalization stats contain NaN/Inf. Regenerate particle_evolution_dataset.npz "
        "with the updated processdata_evolution_u.py finite filtering and field-query bounds."
    )
coord_min = np.asarray(dataset_file["coord_min"], dtype=np.float32).reshape(3) if "coord_min" in dataset_file.files else np.min(inputs_t[:, coord_feature_indices], axis=0)
coord_span = np.asarray(dataset_file["coord_span"], dtype=np.float32).reshape(3) if "coord_span" in dataset_file.files else np.ptp(inputs_t[:, coord_feature_indices], axis=0)
coord_span = np.maximum(coord_span, 1e-8)

delta_position_slice = slice(0, 3)
delta_state_slice = slice(3, 7)
delta_velocity_slice = slice(7, 10)


def as_context(obj) -> Dict:
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "item"):
        item = obj.item()
        if isinstance(item, dict):
            return item
    return dict(obj)


def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    return np.clip((xyz.astype(np.float32) - coord_min[None, :]) / coord_span[None, :], 0.0, 1.0).astype(np.float32)


def sample_indices(n: int, cap: Optional[int], seed: int) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(n, size=int(cap), replace=False)).astype(np.int64)


def sample_indices_random(n: int, cap: Optional[int]) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    return np.sort(np.random.choice(n, size=int(cap), replace=False)).astype(np.int64)

# class EvolutionMultiTaskDataset(Dataset):
#     def __init__(self, pair_ids, split_name, max_input_particles, max_query_points):
#         self.pair_ids = np.asarray(pair_ids, dtype=np.int64)
#         self.split_name = split_name
#         self.max_input_particles = max_input_particles
#         self.max_query_points = max_query_points

#     def __len__(self):
#         return int(len(self.pair_ids))

#     def __getitem__(self, index):
#         pair_id = int(self.pair_ids[int(index)])
#         features_all = np.asarray(inputs_by_pair[pair_id], dtype=np.float32)
#         delta_norm_all = np.asarray(targets_delta_by_pair_norm[pair_id], dtype=np.float32)
#         n = min(features_all.shape[0], delta_norm_all.shape[0])
#         input_idx = sample_indices(n, self.max_input_particles, SEED + pair_id)
#         input_features = features_all[input_idx]
#         x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
#         y_delta = delta_norm_all[input_idx]

#         valid_query_idx = np.flatnonzero(field_query_mask_all[pair_id])
#         if valid_query_idx.size == 0:
#             raise RuntimeError(f"pair_id={pair_id} has no field-grid query points")
#         # The MLP decoder is trained on grid/off-particle queries, not particle positions.
#         # Random sampling here means the decoder sees different grid subsets across epochs.
#         local_query_idx = sample_indices_random(len(valid_query_idx), self.max_query_points)
#         query_idx = valid_query_idx[local_query_idx]
#         query_xyz = query_coords_all[pair_id, query_idx, :]
#         y_field = targets_field_norm_all[pair_id, query_idx, :]
#         context = as_context(frame_contexts[pair_id])
#         return {
#             "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])),
#             "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)),
#             "output_queries": torch.from_numpy(normalize_xyz(query_xyz)),
#             "query_xyz_raw": torch.from_numpy(query_xyz.astype(np.float32)),
#             "y_delta": torch.from_numpy(np.nan_to_num(y_delta).astype(np.float32)),
#             "y_field": torch.from_numpy(np.nan_to_num(y_field).astype(np.float32)),
#             "pair_id": torch.tensor(pair_id, dtype=torch.long),
#             "case": str(context.get("case", "unknown")),
#         }

class EvolutionMultiTaskDataset(Dataset):
    def __init__(self, pair_ids, split_name, max_input_particles, max_query_points,
                 inputs_t, targets_delta_norm_all, query_coords_all,
                 targets_field_norm_all, field_query_mask_all, frame_ranges):
        self.pair_ids = np.asarray(pair_ids, dtype=np.int64)
        self.max_input_particles = max_input_particles
        self.max_query_points = max_query_points

        # Store references to the big arrays, not per‑pair copies
        self.inputs_t = inputs_t
        self.targets_delta_norm_all = targets_delta_norm_all
        self.query_coords_all = query_coords_all
        self.targets_field_norm_all = targets_field_norm_all
        self.field_query_mask_all = field_query_mask_all
        self.frame_ranges = frame_ranges

    def __len__(self):
        return len(self.pair_ids)

    def __getitem__(self, index):
        pair_id = int(self.pair_ids[index])
        start, end = int(self.frame_ranges[pair_id][3]), int(self.frame_ranges[pair_id][4])
        # Slice directly from the big arrays – no per‑pair copies
        features_all = self.inputs_t[start:end]
        delta_norm_all = self.targets_delta_norm_all[start:end]
        n = min(features_all.shape[0], delta_norm_all.shape[0])

        input_idx = sample_indices(n, self.max_input_particles, SEED + pair_id)
        input_features = features_all[input_idx]
        x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
        y_delta = delta_norm_all[input_idx]

        valid_query_idx = np.flatnonzero(self.field_query_mask_all[pair_id])
        if valid_query_idx.size == 0:
            raise RuntimeError(f"pair_id={pair_id} has no field-grid query points")
        local_query_idx = sample_indices_random(len(valid_query_idx), self.max_query_points)
        query_idx = valid_query_idx[local_query_idx]
        query_xyz = self.query_coords_all[pair_id, query_idx, :]
        y_field = self.targets_field_norm_all[pair_id, query_idx, :]

        context = as_context(frame_contexts[pair_id])
        return {
            "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])),
            "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)),
            "output_queries": torch.from_numpy(normalize_xyz(query_xyz)),
            "query_xyz_raw": torch.from_numpy(query_xyz.astype(np.float32)),
            "y_delta": torch.from_numpy(np.nan_to_num(y_delta).astype(np.float32)),
            "y_field": torch.from_numpy(np.nan_to_num(y_field).astype(np.float32)),
            "pair_id": torch.tensor(pair_id, dtype=torch.long),
            "case": str(context.get("case", "unknown")),
        }


def collate_one(batch):
    item = batch[0]
    out = {k: (v.unsqueeze(0) if torch.is_tensor(v) and k != "pair_id" else v) for k, v in item.items()}
    out["pair_id"] = item["pair_id"].view(1)
    return out

train_ds = EvolutionMultiTaskDataset(
    train_pair_ids, "train", CFG["maximum_input_particles"], CFG["maximum_train_query_points"],
    inputs_t, targets_delta_norm_all, query_coords_all, targets_field_norm_all, field_query_mask_all, frame_ranges,
)
val_ds = EvolutionMultiTaskDataset(
    val_pair_ids, "val", CFG["maximum_input_particles"], CFG["maximum_eval_query_points"],
    inputs_t, targets_delta_norm_all, query_coords_all, targets_field_norm_all, field_query_mask_all, frame_ranges,
)
test_ds = EvolutionMultiTaskDataset(
    test_pair_ids, "test", CFG["maximum_input_particles"], CFG["maximum_eval_query_points"],
    inputs_t, targets_delta_norm_all, query_coords_all, targets_field_norm_all, field_query_mask_all, frame_ranges,
)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=CFG["num_workers"], collate_fn=collate_one)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
print("Features:", feature_names)
print("Delta targets:", target_names)
print("Field targets:", field_target_names)
print("Field mean/std:", field_mean.tolist(), field_std.tolist())
if "field_query_bounds" in dataset_file.files:
    print("Field query bounds:", np.asarray(dataset_file["field_query_bounds"]).tolist())
print("Splits:", {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})


In [ ]:
def make_latent_queries(res: int, device: torch.device) -> torch.Tensor:
    line = torch.linspace(0.0, 1.0, int(res), dtype=torch.float32, device=device)
    xx, yy, zz = torch.meshgrid(line, line, line, indexing="ij")
    return torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)

LATENT_QUERIES = make_latent_queries(CFG["latent_res"], DEVICE)


def make_gnoblock(in_channels, out_channels, radius):
    if GNOBlock is None:
        raise RuntimeError("neuralop.layers.gno_block.GNOBlock is not available") from GNO_IMPORT_ERROR
    kwargs = dict(
        in_channels=in_channels,
        out_channels=out_channels,
        coord_dim=3,
        radius=float(radius),
        transform_type="linear",
        reduction="mean",
        pos_embedding_type="transformer",
        pos_embedding_channels=12,
        channel_mlp_layers=[out_channels, out_channels, out_channels],
    )
    accepted = set(inspect.signature(GNOBlock.__init__).parameters)
    if "use_torch_scatter_reduce" in accepted:
        kwargs["use_torch_scatter_reduce"] = False
    if "use_open3d_neighbor_search" in accepted:
        kwargs["use_open3d_neighbor_search"] = False
    return GNOBlock(**{k: v for k, v in kwargs.items() if k in accepted})


def positional_encoding(coords, num_frequencies: int):
    if int(num_frequencies) <= 0:
        return coords
    freqs = (2.0 ** torch.arange(int(num_frequencies), device=coords.device, dtype=coords.dtype)).view(1, 1, -1)
    angles = coords.unsqueeze(-1) * freqs * math.pi
    return torch.cat([coords, torch.sin(angles).flatten(-2), torch.cos(angles).flatten(-2)], dim=-1)


class GINOSharedLatent(nn.Module):
    def __init__(self, in_channels, delta_channels, field_channels, cfg):
        super().__init__()
        hidden = int(cfg["hidden_channels"])
        self.latent_res = int(cfg["latent_res"])
        self.query_pe_freqs = int(cfg["query_pos_encoding_frequencies"])
        self.lift = nn.Sequential(nn.Linear(in_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.encoder = make_gnoblock(hidden, hidden, cfg["gno_radius"])
        if FNO is None:
            raise RuntimeError("neuralop.models.FNO is not available; install/use neuralop with FNO support") from FNO_IMPORT_ERROR
        modes = min(int(cfg["fno_modes"]), max(self.latent_res // 2, 1))
        self.fno = FNO(
            n_modes=(modes, modes, modes),
            in_channels=hidden,
            out_channels=hidden,
            hidden_channels=hidden,
            n_layers=int(cfg["fno_layers"]),
            positional_embedding=None,
        )
        self.delta_decoder_gno = make_gnoblock(hidden, hidden, cfg["gno_radius"])
        self.delta_head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, delta_channels))
        query_dim = 3 + 2 * 3 * self.query_pe_freqs
        mlp_in = hidden + query_dim
        width = int(cfg["mlp_hidden"])
        layers = []
        for layer_id in range(max(int(cfg["mlp_layers"]), 1)):
            layers += [nn.Linear(mlp_in if layer_id == 0 else width, width), nn.GELU()]
        layers.append(nn.Linear(width, field_channels))
        self.field_decoder = nn.Sequential(*layers)
        self.log_delta_var = nn.Parameter(torch.zeros(()))
        self.log_field_var = nn.Parameter(torch.zeros(()))

    def apply_gno(self, block, source_coords, query_coords, source_features):
        # Direction is source points -> query points. In this installed neuralop,
        # GNOBlock names source coordinates `y` and query coordinates `x`.
        return block(y=source_coords, x=query_coords, f_y=source_features)

    def encode_process(self, input_geom, latent_queries, x):
        base_latent = latent_queries[0]
        r = self.latent_res
        grids = []
        flat_latents = []
        for b in range(x.shape[0]):
            h = self.lift(x[b])
            latent = self.apply_gno(self.encoder, source_coords=input_geom[b], query_coords=base_latent, source_features=h)
            if latent.ndim == 3:
                latent = latent.squeeze(0)
            grid = latent.reshape(r, r, r, -1).permute(3, 0, 1, 2).unsqueeze(0)
            processed_grid = self.fno(grid)
            processed_flat = processed_grid.squeeze(0).permute(1, 2, 3, 0).reshape(-1, processed_grid.shape[1])
            grids.append(processed_grid)
            flat_latents.append(processed_flat)
        return base_latent, grids, flat_latents

    def sample_grid(self, grid, queries):
        q = queries.clamp(0.0, 1.0)
        sample_grid = (q * 2.0 - 1.0).view(1, -1, 1, 1, 3)
        sampled = torch.nn.functional.grid_sample(grid, sample_grid, align_corners=True, mode="bilinear")
        return sampled.squeeze(0).squeeze(-1).squeeze(-1).transpose(0, 1)

    def forward(self, input_geom, latent_queries, output_queries, x):
        base_latent, grids, flat_latents = self.encode_process(input_geom, latent_queries, x)
        delta_outputs = []
        field_outputs = []
        for b, (grid, flat_latent) in enumerate(zip(grids, flat_latents)):
            # Delta decoder direction: latent grid -> particle coordinates.
            particle_latent = self.apply_gno(self.delta_decoder_gno, source_coords=base_latent, query_coords=input_geom[b], source_features=flat_latent)
            if particle_latent.ndim == 3:
                particle_latent = particle_latent.squeeze(0)
            delta_outputs.append(self.delta_head(particle_latent))
            q = output_queries[b].clamp(0.0, 1.0)
            sampled = self.sample_grid(grid, q)
            field_outputs.append(self.field_decoder(torch.cat([sampled, positional_encoding(q.unsqueeze(0), self.query_pe_freqs).squeeze(0)], dim=-1)))
        return torch.stack(delta_outputs, dim=0), torch.stack(field_outputs, dim=0)

model = GINOSharedLatent(len(feature_names), len(target_names), len(field_target_names), CFG).to(DEVICE)
target_mean_t = torch.tensor(target_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_std_t = torch.tensor(target_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
field_mean_t = torch.tensor(field_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
field_std_t = torch.tensor(field_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
print(model)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
def move_batch(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}


def predict(batch):
    return model(batch["input_geom"], LATENT_QUERIES, batch["output_queries"], batch["x"])


def denormalize_delta(y):
    return y * target_std_t + target_mean_t


def denormalize_field(y):
    return y * field_std_t + field_mean_t


def relative_l2(pred, target, eps=1e-8):
    pred = torch.nan_to_num(pred)
    target = torch.nan_to_num(target)
    numerator = torch.linalg.norm((pred - target).reshape(pred.shape[0], -1), dim=1)
    denominator = torch.linalg.norm(target.reshape(target.shape[0], -1), dim=1).clamp_min(eps)
    return numerator / denominator


def multitask_loss(delta_pred, field_pred, y_delta, y_field):
    loss_delta = F.mse_loss(delta_pred, y_delta)
    loss_field = F.mse_loss(field_pred, y_field)
    if CFG["loss_weighting"].lower() == "uncertainty":
        total = torch.exp(-model.log_delta_var) * loss_delta + model.log_delta_var
        total = total + torch.exp(-model.log_field_var) * loss_field + model.log_field_var
    else:
        total = loss_delta + float(CFG["field_loss_weight"]) * loss_field
    return total, loss_delta.detach(), loss_field.detach()


@torch.no_grad()
def evaluate(loader, max_batches=None):
    model.eval()
    if len(loader.dataset) == 0:
        return {"loss": math.nan, "delta_loss": math.nan, "field_loss": math.nan, "delta_rel_l2": math.nan, "field_rel_l2": math.nan, "batches": 0}
    losses, delta_losses, field_losses, delta_rels, field_rels = [], [], [], [], []
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= int(max_batches):
            break
        batch = move_batch(batch)
        delta_pred, field_pred = predict(batch)
        loss, delta_loss, field_loss = multitask_loss(delta_pred, field_pred, batch["y_delta"], batch["y_field"])
        delta_pred_phys = denormalize_delta(delta_pred)
        y_delta_phys = denormalize_delta(batch["y_delta"])
        losses.append(float(loss.item()))
        delta_losses.append(float(delta_loss.item()))
        field_losses.append(float(field_loss.item()))
        delta_rels.append(float(relative_l2(delta_pred_phys, y_delta_phys).mean().item()))
        # Field relative L2 is measured in normalized space so the metric focuses
        # on the learnable fluctuation signal instead of the freestream offset.
        field_rels.append(float(relative_l2(field_pred, batch["y_field"]).mean().item()))
    return {
        "loss": float(np.mean(losses)) if losses else math.nan,
        "delta_loss": float(np.mean(delta_losses)) if delta_losses else math.nan,
        "field_loss": float(np.mean(field_losses)) if field_losses else math.nan,
        "delta_rel_l2": float(np.mean(delta_rels)) if delta_rels else math.nan,
        "field_rel_l2": float(np.mean(field_rels)) if field_rels else math.nan,
        "batches": len(losses),
    }

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CFG["epochs"], 1))
history = []
best = float("inf")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses, delta_losses, field_losses = [], [], []
    accum = max(int(CFG["gradient_accumulation_steps"]), 1)
    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        delta_pred, field_pred = predict(batch)
        loss, delta_loss, field_loss = multitask_loss(delta_pred, field_pred, batch["y_delta"], batch["y_field"])
        (loss / accum).backward()
        if step % accum == 0 or step == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.item()))
        delta_losses.append(float(delta_loss.item()))
        field_losses.append(float(field_loss.item()))
    scheduler.step()
    if epoch % CFG["eval_every"] == 0 or epoch == CFG["epochs"]:
        val = evaluate(val_loader, max_batches=CFG["maximum_eval_batches"])
        test = evaluate(test_loader, max_batches=min(8, CFG["maximum_eval_batches"] or 8))
        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_delta_loss": float(np.mean(delta_losses)),
            "train_field_loss": float(np.mean(field_losses)),
            "log_delta_var": float(model.log_delta_var.detach().cpu()),
            "log_field_var": float(model.log_field_var.detach().cpu()),
            "val": val,
            "test": test,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(json.dumps(row, indent=2))
        score = val["field_rel_l2"] if np.isfinite(val["field_rel_l2"]) else row["train_loss"]
        if score < best:
            best = score
            torch.save({
                "checkpoint_tag": RUN_TAG,
                "saved_at_utc": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
                "config": CFG,
                "model_state_dict": model.state_dict(),
                "feature_names_all": feature_names_all,
                "feature_names": feature_names,
                "target_names": target_names,
                "field_target_names": field_target_names,
                "active_input_feature_indices": active_input_feature_indices,
                "coord_feature_indices": coord_feature_indices,
                "coord_min": coord_min,
                "coord_span": coord_span,
                "input_mean": input_mean,
                "input_std": input_std,
                "target_mean": target_mean,
                "target_std": target_std,
                "field_mean": field_mean,
                "field_std": field_std,
                "dataset_path": str(DATASET_PATH),
                "best_score": best,
                "history": history,
            }, CKPT_PATH)
            print("saved", CKPT_PATH)
HISTORY_PATH.write_text(json.dumps(history, indent=2))


In [ ]:
@torch.no_grad()
def predict_u_on_field_grid(pair_id: int, grid_resolution: Optional[int] = None, y_value: Optional[float] = None, max_input_particles: Optional[int] = None):
    """Query the MLP field decoder on an x-z plane across the full coordinate extent."""
    model.eval()
    grid_resolution = int(grid_resolution or CFG["sr_grid_resolution"])
    pair_range = frame_ranges[int(pair_id)]
    start, end = int(pair_range[3]), int(pair_range[4])
    features_all = inputs_t[start:end]
    input_idx = sample_indices(features_all.shape[0], max_input_particles or CFG["maximum_input_particles"], SEED + int(pair_id))
    input_features = features_all[input_idx]
    x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
    x_line = np.linspace(coord_min[0], coord_min[0] + coord_span[0], grid_resolution, dtype=np.float32)
    z_line = np.linspace(coord_min[2], coord_min[2] + coord_span[2], grid_resolution, dtype=np.float32)
    xx, zz = np.meshgrid(x_line, z_line, indexing="xy")
    if y_value is None:
        y_value = float(np.median(features_all[:, coord_feature_indices[1]]))
    yy = np.full_like(xx, float(y_value), dtype=np.float32)
    query_xyz = np.stack([xx, yy, zz], axis=-1).reshape(-1, 3)
    batch = {
        "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])).unsqueeze(0).to(DEVICE),
        "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)).unsqueeze(0).to(DEVICE),
        "output_queries": torch.from_numpy(normalize_xyz(query_xyz)).unsqueeze(0).to(DEVICE),
    }
    _, field_pred_norm = predict(batch)
    u = denormalize_field(field_pred_norm).squeeze(0).cpu().numpy()
    return query_xyz, u, u.reshape(grid_resolution, grid_resolution, 3)

if len(test_ds) > 0:
    pair_id = int(test_ds.pair_ids[0])
elif len(val_ds) > 0:
    pair_id = int(val_ds.pair_ids[0])
else:
    pair_id = int(train_ds.pair_ids[0])
query_xyz, u_flat, u_grid = predict_u_on_field_grid(pair_id, grid_resolution=min(CFG["sr_grid_resolution"], 96))
u_mag = np.linalg.norm(u_grid, axis=-1)
fig, ax = plt.subplots(figsize=(7.2, 5.5), constrained_layout=True)
im = ax.imshow(u_mag, origin="lower", extent=[query_xyz[:,0].min(), query_xyz[:,0].max(), query_xyz[:,2].min(), query_xyz[:,2].max()], aspect="auto")
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.set_title(f"MLP field decoder query |u|, pair_id={pair_id}")
plt.colorbar(im, ax=ax, label="|u|")
plot_path = RESULTS_DIR / f"{RUN_TAG}_sr_u_field.png"
fig.savefig(plot_path, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)
